# Grid Search — Compare Models on the Same Task

Find the best (generation model, reflection model) pair by running a full optimization for every combination.

**Dataset:** `wikidata_qa.json` — 100 hand-curated factual questions sourced from Wikidata (CC0).

**What you'll learn:**
1. Submit a grid search (one job, multiple model pairs)
2. Monitor per-pair progress
3. Retrieve and compare grid results
4. Load the best program

**Prerequisites:** Complete [01_quickstart.ipynb](01_quickstart.ipynb) first.

In [ ]:
import json
import os
from pathlib import Path

import dspy

from skynet_client import DSPyServiceClient, JobMonitor, serialize_source

In [ ]:
BASE_URL = os.getenv("DSPY_SERVICE_URL", "http://localhost:8000")
LM_BASE_URL = os.getenv("DSPY_LM_BASE_URL", "https://api.openai.com/v1")

# LM_API_KEY is the canonical name; OPENAI_API_KEY is accepted as a
# fallback for backwards-compat and the public OpenAI dev path.
LM_API_KEY = os.getenv("LM_API_KEY") or os.getenv("OPENAI_API_KEY")
if not LM_API_KEY:
    raise ValueError("Set LM_API_KEY (or OPENAI_API_KEY) — any non-empty token works for self-hosted gateways.")

MODEL_CONFIG = {
    "name": "openai/gpt-4o-mini",
    "base_url": LM_BASE_URL,
    "model_type": "responses",
    "temperature": 1.0,
    "max_tokens": 20000,
    "extra": {"api_key": LM_API_KEY},
}

dspy.configure(
    lm=dspy.LM(
        "openai/gpt-4o-mini",
        model_type="responses",
        temperature=1.0,
        max_tokens=20000,
        api_base=LM_BASE_URL,
        api_key=LM_API_KEY,
    )
)

client = DSPyServiceClient(BASE_URL)
client.health()

## 1. Dataset & Signature

In [ ]:
with open(Path("../../data/wikidata_qa.json")) as f:
    dataset = json.load(f)

print(f"{len(dataset)} examples")
print(f"Sample: {dataset[0]}")

In [ ]:
class FactualQA(dspy.Signature):
    """Answer general-domain factual questions with a short, exact phrase."""
    question: str = dspy.InputField(desc="A factual question with a short, definite answer")
    answer: str = dspy.OutputField(desc="A concise answer — usually a name, place, or number")


SIGNATURE_CODE = serialize_source(FactualQA)

In [ ]:
# Token-overlap F1: rewards partial credit for paraphrases like
# "Mount Everest" vs. "Everest" without requiring exact string match.
METRIC_CODE = '''
def factual_qa_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    import re

    def tokens(text):
        return set(re.findall(r"[a-z0-9]+", (text or "").lower()))

    g = tokens(gold.answer)
    p = tokens(pred.answer)

    if not g or not p:
        return dspy.Prediction(score=0.0, feedback="Empty answer.")

    overlap = g & p
    precision = len(overlap) / len(p)
    recall = len(overlap) / len(g)
    f1 = (2 * precision * recall / (precision + recall)) if overlap else 0.0
    f1 = round(f1, 2)

    if f1 == 1.0:
        return dspy.Prediction(score=1.0, feedback="Exact token match.")
    elif f1 > 0:
        msg = f"Partial. Expected \"{gold.answer}\", got \"{pred.answer}\" (F1={f1})."
        return dspy.Prediction(score=f1, feedback=msg)
    else:
        msg = f"Wrong. Expected \"{gold.answer}\", got \"{pred.answer}\". Be precise and concise."
        return dspy.Prediction(score=0.0, feedback=msg)
'''

## 2. Define the Model Grid

Grid search replaces `model_config` + `reflection_model_config` with two lists.
The service runs one full optimization for every `(generation, reflection)` pair.

Below we compare two generation models with one reflection model = 2 pairs.

In [ ]:
generation_models = [
    {
        "name": "openai/gpt-4o-mini",
        "base_url": LM_BASE_URL,
        "model_type": "responses",
        "temperature": 0.7,
        "max_tokens": 4000,
        "extra": {"api_key": LM_API_KEY},
    },
    {
        "name": "openai/gpt-4o",
        "base_url": LM_BASE_URL,
        "model_type": "responses",
        "temperature": 0.7,
        "max_tokens": 4000,
        "extra": {"api_key": LM_API_KEY},
    },
]

reflection_models = [
    {
        "name": "openai/gpt-4o-mini",
        "base_url": LM_BASE_URL,
        "model_type": "responses",
        "temperature": 1.0,
        "max_tokens": 20000,
        "extra": {"api_key": LM_API_KEY},
    },
]

print(f"Grid: {len(generation_models)} x {len(reflection_models)} = {len(generation_models) * len(reflection_models)} pairs")

## 3. Submit Grid Search

In [ ]:
payload = {
    "username": "notebook-user",
    "module_name": "dspy.ChainOfThought",
    "signature_code": SIGNATURE_CODE,
    "metric_code": METRIC_CODE,
    "optimizer_name": "dspy.GEPA",
    "optimizer_kwargs": {"auto": "light", "num_threads": 8},
    "compile_kwargs": {},
    "dataset": dataset,
    "column_mapping": {
        "inputs": {"question": "question"},
        "outputs": {"answer": "answer"},
    },
    "split_fractions": {"train": 0.5, "val": 0.3, "test": 0.2},
    "shuffle": True,
    "seed": 42,
    "generation_models": generation_models,
    "reflection_models": reflection_models,
}

optimization_id = client.submit_grid_search(payload)
print(f"Submitted grid search: {optimization_id}")

## 4. Monitor

Grid search jobs use the same monitoring API. Progress events include per-pair starts/completions.

In [ ]:
monitor = JobMonitor(client, optimization_id)
result = monitor.poll(interval=10)

print(f"\nFinal status: {result['status']}")

## 5. Compare Results

The `/grid-result` endpoint returns per-pair metrics and identifies the best combination.

In [ ]:
grid = client.grid_result(optimization_id)

print(f"Completed: {grid['completed_pairs']}/{grid['total_pairs']}")
print(f"Failed:    {grid['failed_pairs']}")
print()

In [ ]:
# Per-pair breakdown
for i, pair in enumerate(grid["pair_results"]):
    status = "FAILED" if pair.get("error") else "OK"
    gen = pair["generation_model"]["name"]
    ref = pair["reflection_model"]["name"]
    baseline = pair.get("baseline_test_metric", "N/A")
    optimized = pair.get("optimized_test_metric", "N/A")
    improvement = pair.get("metric_improvement", "N/A")
    print(f"Pair {i}: {gen} + {ref}")
    print(f"  Baseline={baseline}  Optimized={optimized}  Improvement={improvement}  [{status}]")

In [ ]:
# Best pair
if grid["best_pair"]:
    bp = grid["best_pair"]
    print(f"Best: {bp['generation_model']['name']} + {bp['reflection_model']['name']}")
    print(f"  Score: {bp['optimized_test_metric']}")
    print(f"  Improvement: {bp['metric_improvement']}")

## 6. Load the Best Program

In [ ]:
program = client.load_program(optimization_id)
print(f"Loaded: {type(program).__name__}")

In [ ]:
test_questions = [
    "Which country has Reykjavik as its capital?",
    "Who composed the ballet 'The Nutcracker'?",
    "What is the capital of Mongolia?",
]

for q in test_questions:
    response = program(question=q)
    print(f"Q: {q}")
    print(f"A: {response.answer}\n")

## Filtering Logs by Level

The logs endpoint supports level filtering and pagination — useful for debugging failed pairs.

In [ ]:
error_logs = client.logs(optimization_id, level="ERROR")
print(f"{len(error_logs)} error logs")
for log in error_logs[:5]:
    print(f"  {log['message'][:100]}")

## Per-Example Test Results

See how baseline vs. optimized performed on each test example.

In [ ]:
test_results = client.test_results(optimization_id)

baseline_scores = test_results.get("baseline", [])
optimized_scores = test_results.get("optimized", [])
print(f"{len(baseline_scores)} test examples")

for i, (b, o) in enumerate(zip(baseline_scores[:5], optimized_scores[:5], strict=True)):
    b_score = b.get("score", "N/A")
    o_score = o.get("score", "N/A")
    marker = "+" if o_score > b_score else ("=" if o_score == b_score else "-")
    print(f"  Example {i}: baseline={b_score}  optimized={o_score}  [{marker}]")

## What's Next

- **[03_creative_tasks.ipynb](03_creative_tasks.ipynb)** — Multi-output structured prediction over NASA missions.
- **API Reference** — `http://localhost:8000/reference`